In [1]:
# Cell 1. Pre-sleep Design C Stage 2 training setup and tensor validation
# 원하는 결과:
# - Stage 2 rolling/history final MLP tensor를 로드한다.
# - train/validation/test shape, target distribution, NaN/Inf를 확인한다.
# - Stage 1 결과와 비교 기준 경로를 준비한다.
# - 학습 실행 flag는 기본 False로 둔다.

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STAGE2_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage2_rolling_history"
TENSOR_DIR = STAGE2_DIR / "mlp_current_day_final"
METADATA_PATH = STAGE2_DIR / "metadata.json"
FEATURE_COLUMNS_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_final_feature_columns.csv"

STAGE1_OUTPUT_DIR = (
    PROCESSED_DIR
    / "pre_sleep_forecasting"
    / "design_c_stage1"
    / "experiments"
    / "stage1_mlp_outputs"
)
STAGE1_METRICS_PATH = STAGE1_OUTPUT_DIR / "pre_sleep_stage1_mlp_metrics.csv"

OUTPUT_DIR = STAGE2_DIR / "experiments" / "stage2_mlp_outputs"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

RUN_PRE_SLEEP_STAGE2_TRAINING = False

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "presleep_stage2_000"
MODEL_FAMILY = "mlp_current_day"
FEATURE_TIMING = "pre_sleep"
SUBSET_NAME = "design_c_stage2_rolling_history"
WINDOW = 1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def load_npz_split(split_name):
    path = TENSOR_DIR / f"{split_name}.npz"
    data = np.load(path, allow_pickle=True)
    return {
        "path": path,
        "X": data["X"].astype(np.float32),
        "y": data["y"].astype(np.int64),
        "sleep_episode_id": data["sleep_episode_id"].astype(str),
        "participant_object_id": data["participant_object_id"].astype(str),
        "feature_columns": data["feature_columns"].astype(str),
    }

split_data = {
    split_name: load_npz_split(split_name)
    for split_name in ["train", "validation", "test"]
}

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
feature_columns_df = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig")

loaded_feature_columns = split_data["train"]["feature_columns"].tolist()

tensor_summary_rows = []

for split_name, data in split_data.items():
    X = data["X"]
    y = data["y"]
    participant_ids = data["participant_object_id"]

    tensor_summary_rows.append(
        {
            "split": split_name,
            "path": str(data["path"].relative_to(PROJECT_ROOT)),
            "X_shape": X.shape,
            "samples": int(X.shape[0]),
            "features": int(X.shape[1]),
            "participants": int(pd.Series(participant_ids).nunique()),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

tensor_summary_df = pd.DataFrame(tensor_summary_rows)

stage1_metrics_df = pd.read_csv(STAGE1_METRICS_PATH, encoding="utf-8-sig")

feature_check = pd.DataFrame(
    [
        {"metric": "metadata_final_feature_count", "value": metadata["final_feature_count"]},
        {"metric": "feature_columns_csv_rows", "value": len(feature_columns_df)},
        {"metric": "npz_feature_count", "value": len(loaded_feature_columns)},
        {"metric": "train_X_feature_count", "value": split_data["train"]["X"].shape[1]},
    ]
)

problem_rows = tensor_summary_df[
    (tensor_summary_df["nan_count"] > 0)
    | (tensor_summary_df["inf_count"] > 0)
]

stage1_test_row = stage1_metrics_df[stage1_metrics_df["split"] == "test"].copy()

print("=== Pre-Sleep Stage 2 Training Setup ===")
print("DEVICE:", DEVICE)
print("RUN_PRE_SLEEP_STAGE2_TRAINING:", RUN_PRE_SLEEP_STAGE2_TRAINING)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("TENSOR_DIR:", TENSOR_DIR)
print("STAGE1_METRICS_PATH:", STAGE1_METRICS_PATH, STAGE1_METRICS_PATH.exists())

print("\n=== Tensor Summary ===")
display(tensor_summary_df)

print("\n=== Feature Check ===")
display(feature_check)

print("\n=== Feature Group Counts ===")
display(feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Stage 1 Test Reference ===")
display(
    stage1_test_row[
        [
            "experiment_id",
            "feature_timing",
            "subset_name",
            "model_family",
            "split",
            "balanced_accuracy",
            "roc_auc",
            "average_precision",
            "f1",
            "precision",
            "recall",
        ]
    ]
)

print("\n=== Problem Rows ===")
if len(problem_rows) > 0:
    display(problem_rows)
else:
    print("없음")

print("\nReady for next step: define Stage 2 MLP model and training utilities.")

=== Pre-Sleep Stage 2 Training Setup ===
DEVICE: cpu
RUN_PRE_SLEEP_STAGE2_TRAINING: False
EXPERIMENT_ID: presleep_stage2_000
TENSOR_DIR: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\mlp_current_day_final
STAGE1_METRICS_PATH: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\pre_sleep_stage1_mlp_metrics.csv True

=== Tensor Summary ===


,split,path,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,train,data\processed\pre_sleep_forecasting\design_c_...,"(2323, 380)",2323,380,46,1365,958,0.412398,0,0
1,validation,data\processed\pre_sleep_forecasting\design_c_...,"(347, 380)",347,380,9,225,122,0.351585,0,0
2,test,data\processed\pre_sleep_forecasting\design_c_...,"(881, 380)",881,380,14,563,318,0.360953,0,0



=== Feature Check ===


,metric,value
0,metadata_final_feature_count,380
1,feature_columns_csv_rows,380
2,npz_feature_count,380
3,train_X_feature_count,380



=== Feature Group Counts ===


,feature_group,features
0,rolling_history,319
1,stage1_final,58
2,history_indicator,3



=== Stage 1 Test Reference ===


,experiment_id,feature_timing,subset_name,model_family,split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall
2,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,test,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516



=== Problem Rows ===
없음

Ready for next step: define Stage 2 MLP model and training utilities.


In [2]:
# Cell 2. Define Stage 2 MLP model, training loop, and evaluation utilities
# 원하는 결과:
# - rolling/history feature 380개 입력을 받는 MLP 모델 구조를 정의한다.
# - validation balanced accuracy 기준 early stopping 학습 함수를 준비한다.
# - threshold search/evaluation 함수를 준비한다.
# - 아직 학습은 하지 않는다.

class PreSleepStage2MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64, 32), dropout=0.35):
        super().__init__()

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(current_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

def make_loader(X, y, batch_size=64, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def predict_proba(model, X, batch_size=256):
    model.eval()
    probabilities = []

    loader = make_loader(X, np.zeros(X.shape[0]), batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch_X, _ in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)
            batch_probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            probabilities.append(batch_probabilities)

    return np.concatenate(probabilities)

def evaluate_binary_predictions(y_true, y_probability, threshold):
    y_pred = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probability)
        metrics["average_precision"] = average_precision_score(y_true, y_probability)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics

def find_best_threshold(y_true, y_probability, metric_name="balanced_accuracy"):
    threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []
    for threshold in threshold_grid:
        row = evaluate_binary_predictions(y_true, y_probability, threshold)
        rows.append(row)

    threshold_df = pd.DataFrame(rows)

    best_row = (
        threshold_df
        .sort_values(
            [metric_name, "f1", "balanced_accuracy"],
            ascending=[False, False, False],
        )
        .iloc[0]
        .to_dict()
    )

    return float(best_row["threshold"]), threshold_df, best_row

def train_one_stage2_model(
    experiment_id,
    X_train,
    y_train,
    X_validation,
    y_validation,
    input_dim,
    seed=2026,
    max_epochs=120,
    patience=20,
    batch_size=64,
    learning_rate=8e-4,
    weight_decay=3e-4,
    hidden_dims=(128, 64, 32),
    dropout=0.35,
):
    set_seed(seed)

    model = PreSleepStage2MLP(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(DEVICE)

    positive_count = float((y_train == 1).sum())
    negative_count = float((y_train == 0).sum())
    pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = None
    best_threshold = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        validation_probability = predict_proba(model, X_validation)
        threshold, threshold_df, best_threshold_row = find_best_threshold(
            y_validation,
            validation_probability,
            metric_name="balanced_accuracy",
        )

        validation_metrics = evaluate_binary_predictions(
            y_validation,
            validation_probability,
            threshold,
        )

        mean_train_loss = float(np.mean(train_losses))

        history_row = {
            "experiment_id": experiment_id,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "validation_selected_threshold": threshold,
            **{
                f"validation_{key}": value
                for key, value in validation_metrics.items()
            },
        }
        history_rows.append(history_row)

        current_validation_balanced_accuracy = validation_metrics["balanced_accuracy"]

        if current_validation_balanced_accuracy > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_validation_balanced_accuracy
            best_epoch = epoch
            best_threshold = threshold
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{experiment_id} epoch {epoch:03d} | "
                f"loss={mean_train_loss:.4f} | "
                f"val_bal_acc={validation_metrics['balanced_accuracy']:.4f} | "
                f"val_auc={validation_metrics['roc_auc']:.4f} | "
                f"thr={threshold:.2f}"
            )

        if epochs_without_improvement >= patience:
            print(
                f"{experiment_id} early stopped at epoch {epoch}; "
                f"best_epoch={best_epoch}, "
                f"best_val_bal_acc={best_validation_balanced_accuracy:.4f}"
            )
            break

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)

    return {
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
        "best_validation_balanced_accuracy": best_validation_balanced_accuracy,
    }

print("Stage 2 training utilities are ready.")
print("Model:", PreSleepStage2MLP(input_dim=split_data["train"]["X"].shape[1]))
print("Next step: set RUN_PRE_SLEEP_STAGE2_TRAINING = True in Cell 3 to train manually.")

Stage 2 training utilities are ready.
Model: PreSleepStage2MLP(
  (network): Sequential(
    (0): Linear(in_features=380, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.35, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.35, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.35, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)
Next step: set RUN_PRE_SLEEP_STAGE2_TRAINING = True in Cell 3 to train manually.


In [4]:
# Cell 3. Train Pre-sleep Design C Stage 2 MLP
# 원하는 결과:
# - Stage 2 rolling/history MLP를 학습한다.
# - validation balanced accuracy 기준 best epoch/threshold를 선택한다.
# - validation-selected threshold로 train/validation/test metrics를 저장한다.
# - Stage 1 test 결과와 비교한다.
# - prediction/history/model 파일을 저장하고 log를 업데이트한다.

RUN_PRE_SLEEP_STAGE2_TRAINING = True

MAX_EPOCHS = 120
PATIENCE = 20
BATCH_SIZE = 64
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 3e-4
HIDDEN_DIMS = (128, 64, 32)
DROPOUT = 0.35

METRICS_PATH = OUTPUT_DIR / "pre_sleep_stage2_mlp_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "pre_sleep_stage2_mlp_training_history.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "pre_sleep_stage2_mlp_predictions.csv"
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_ID}_best.pt"
COMPARISON_PATH = OUTPUT_DIR / "pre_sleep_stage1_vs_stage2_comparison.csv"

if not RUN_PRE_SLEEP_STAGE2_TRAINING:
    print("RUN_PRE_SLEEP_STAGE2_TRAINING is False.")
    print("Set RUN_PRE_SLEEP_STAGE2_TRAINING = True to train manually.")
else:
    print("=== Pre-Sleep Stage 2 MLP Training Started ===")
    print("device:", DEVICE)
    print("experiment_id:", EXPERIMENT_ID)
    print("max_epochs:", MAX_EPOCHS)
    print("patience:", PATIENCE)
    print("batch_size:", BATCH_SIZE)
    print("learning_rate:", LEARNING_RATE)
    print("weight_decay:", WEIGHT_DECAY)
    print("hidden_dims:", HIDDEN_DIMS)
    print("dropout:", DROPOUT)

    train_result = train_one_stage2_model(
        experiment_id=EXPERIMENT_ID,
        X_train=split_data["train"]["X"],
        y_train=split_data["train"]["y"],
        X_validation=split_data["validation"]["X"],
        y_validation=split_data["validation"]["y"],
        input_dim=split_data["train"]["X"].shape[1],
        seed=SEED,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    )

    model = train_result["model"]
    best_epoch = train_result["best_epoch"]
    selected_threshold = train_result["best_threshold"]
    history_df = train_result["history_df"]

    torch.save(
        {
            "experiment_id": EXPERIMENT_ID,
            "model_family": MODEL_FAMILY,
            "feature_timing": FEATURE_TIMING,
            "subset_name": SUBSET_NAME,
            "window": WINDOW,
            "seed": SEED,
            "input_dim": split_data["train"]["X"].shape[1],
            "hidden_dims": HIDDEN_DIMS,
            "dropout": DROPOUT,
            "best_epoch": best_epoch,
            "selected_threshold_from_validation": selected_threshold,
            "state_dict": model.state_dict(),
            "feature_columns": loaded_feature_columns,
        },
        MODEL_PATH,
    )

    metrics_rows = []
    prediction_rows = []

    for split_name in ["train", "validation", "test"]:
        X_split = split_data[split_name]["X"]
        y_split = split_data[split_name]["y"]
        probability = predict_proba(model, X_split)

        metrics = evaluate_binary_predictions(
            y_split,
            probability,
            selected_threshold,
        )

        metrics_rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "feature_timing": FEATURE_TIMING,
                "subset_name": SUBSET_NAME,
                "model_family": MODEL_FAMILY,
                "window": WINDOW,
                "split": split_name,
                "seed": SEED,
                "best_epoch": best_epoch,
                "selected_threshold_from_validation": selected_threshold,
                **metrics,
            }
        )

        split_prediction_df = pd.DataFrame(
            {
                "experiment_id": EXPERIMENT_ID,
                "split": split_name,
                "sleep_episode_id": split_data[split_name]["sleep_episode_id"],
                "participant_object_id": split_data[split_name]["participant_object_id"],
                "y_true": y_split,
                "y_probability": probability,
                "selected_threshold_from_validation": selected_threshold,
                "y_pred": (probability >= selected_threshold).astype(int),
            }
        )
        prediction_rows.append(split_prediction_df)

    metrics_df = pd.DataFrame(metrics_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    stage1_test_reference = stage1_metrics_df[
        stage1_metrics_df["split"] == "test"
    ].iloc[0]

    stage2_test_row = metrics_df[metrics_df["split"] == "test"].iloc[0]

    comparison_df = pd.DataFrame(
        [
            {
                "model_role": "stage1_reference",
                "experiment_id": stage1_test_reference["experiment_id"],
                "subset_name": stage1_test_reference["subset_name"],
                "features": 58,
                "test_balanced_accuracy": stage1_test_reference["balanced_accuracy"],
                "test_roc_auc": stage1_test_reference["roc_auc"],
                "test_average_precision": stage1_test_reference["average_precision"],
                "test_f1": stage1_test_reference["f1"],
                "test_precision": stage1_test_reference["precision"],
                "test_recall": stage1_test_reference["recall"],
            },
            {
                "model_role": "stage2_rolling_history",
                "experiment_id": EXPERIMENT_ID,
                "subset_name": SUBSET_NAME,
                "features": split_data["train"]["X"].shape[1],
                "test_balanced_accuracy": stage2_test_row["balanced_accuracy"],
                "test_roc_auc": stage2_test_row["roc_auc"],
                "test_average_precision": stage2_test_row["average_precision"],
                "test_f1": stage2_test_row["f1"],
                "test_precision": stage2_test_row["precision"],
                "test_recall": stage2_test_row["recall"],
            },
        ]
    )

    comparison_df["delta_balanced_accuracy_vs_stage1"] = (
        comparison_df["test_balanced_accuracy"]
        - float(stage1_test_reference["balanced_accuracy"])
    )

    metrics_df.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
    history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
    comparison_df.to_csv(COMPARISON_PATH, index=False, encoding="utf-8-sig")

    metadata["stage2_training"] = {
        "experiment_id": EXPERIMENT_ID,
        "model_family": MODEL_FAMILY,
        "feature_timing": FEATURE_TIMING,
        "subset_name": SUBSET_NAME,
        "window": WINDOW,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "hidden_dims": list(HIDDEN_DIMS),
        "dropout": DROPOUT,
        "best_epoch": int(best_epoch),
        "selected_threshold_from_validation": float(selected_threshold),
        "metrics_path": str(METRICS_PATH.relative_to(PROJECT_ROOT)),
        "history_path": str(HISTORY_PATH.relative_to(PROJECT_ROOT)),
        "predictions_path": str(PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
        "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
        "comparison_path": str(COMPARISON_PATH.relative_to(PROJECT_ROOT)),
    }
    METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 2 MLP training

Trained rolling/history strict pre-sleep forecasting MLP.

- Notebook: `notebooks/14_pre_sleep_stage2_training.ipynb`
- Experiment: `{EXPERIMENT_ID}`
- Objective: predict `good_sleep_label` using pre-sleep features plus prior episode rolling/history features
- Model: `{MODEL_FAMILY}`
- Features: {split_data["train"]["X"].shape[1]}
- Train/validation/test samples: {split_data["train"]["X"].shape[0]} / {split_data["validation"]["X"].shape[0]} / {split_data["test"]["X"].shape[0]}
- Best epoch: {best_epoch}
- Selected threshold from validation: {selected_threshold:.2f}
- Test balanced accuracy: {stage2_test_row["balanced_accuracy"]:.4f}
- Test ROC AUC: {stage2_test_row["roc_auc"]:.4f}
- Test average precision: {stage2_test_row["average_precision"]:.4f}
- Test F1: {stage2_test_row["f1"]:.4f}
- Test precision: {stage2_test_row["precision"]:.4f}
- Test recall: {stage2_test_row["recall"]:.4f}
- Delta test balanced accuracy vs Stage 1 single-seed: {stage2_test_row["balanced_accuracy"] - float(stage1_test_reference["balanced_accuracy"]):.4f}
- Metrics: `{METRICS_PATH.relative_to(PROJECT_ROOT)}`
- History: `{HISTORY_PATH.relative_to(PROJECT_ROOT)}`
- Predictions: `{PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}`
- Comparison: `{COMPARISON_PATH.relative_to(PROJECT_ROOT)}`
- Model: `{MODEL_PATH.relative_to(PROJECT_ROOT)}`
"""

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(log_entry)

    print("\n=== Pre-Sleep Stage 2 MLP Training Finished ===")
    print("model:", MODEL_PATH.relative_to(PROJECT_ROOT), MODEL_PATH.exists())
    print("metrics:", METRICS_PATH.relative_to(PROJECT_ROOT), METRICS_PATH.exists())
    print("history:", HISTORY_PATH.relative_to(PROJECT_ROOT), HISTORY_PATH.exists())
    print("predictions:", PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PREDICTIONS_PATH.exists())
    print("comparison:", COMPARISON_PATH.relative_to(PROJECT_ROOT), COMPARISON_PATH.exists())

    print("\n=== Metrics ===")
    display(metrics_df)

    print("\n=== Stage 1 vs Stage 2 Test Comparison ===")
    display(comparison_df)

    print("\n=== Test Row ===")
    display(metrics_df.loc[metrics_df["split"] == "test"])

    print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Pre-Sleep Stage 2 MLP Training Started ===
device: cpu
experiment_id: presleep_stage2_000
max_epochs: 120
patience: 20
batch_size: 64
learning_rate: 0.0008
weight_decay: 0.0003
hidden_dims: (128, 64, 32)
dropout: 0.35
presleep_stage2_000 epoch 001 | loss=0.8097 | val_bal_acc=0.6705 | val_auc=0.7119 | thr=0.48
presleep_stage2_000 epoch 010 | loss=0.6290 | val_bal_acc=0.6609 | val_auc=0.7132 | thr=0.39
presleep_stage2_000 epoch 020 | loss=0.5320 | val_bal_acc=0.6559 | val_auc=0.7023 | thr=0.47
presleep_stage2_000 early stopped at epoch 22; best_epoch=2, best_val_bal_acc=0.6936

=== Pre-Sleep Stage 2 MLP Training Finished ===
model: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\experiments\stage2_mlp_outputs\models\presleep_stage2_000_best.pt True
metrics: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_history\experiments\stage2_mlp_outputs\pre_sleep_stage2_mlp_metrics.csv True
history: data\processed\pre_sleep_forecasting\design_c_stage2_rolling_h

,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
0,presleep_stage2_000,pre_sleep,design_c_stage2_rolling_history,mlp_current_day,1,train,2026,2,0.48,0.48,0.672742,0.645852,0.563863,0.755741,805,560,234,724,0.752605,0.671766
1,presleep_stage2_000,pre_sleep,design_c_stage2_rolling_history,mlp_current_day,1,validation,2026,2,0.48,0.48,0.693588,0.627692,0.502463,0.836066,124,101,20,102,0.732969,0.569084
2,presleep_stage2_000,pre_sleep,design_c_stage2_rolling_history,mlp_current_day,1,test,2026,2,0.48,0.48,0.602486,0.530719,0.454139,0.638365,319,244,115,203,0.662779,0.585455



=== Stage 1 vs Stage 2 Test Comparison ===


,model_role,experiment_id,subset_name,features,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,delta_balanced_accuracy_vs_stage1
0,stage1_reference,presleep_stage1_000,design_c_stage1_intraday_previous_day,58,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516,0.000000
1,stage2_rolling_history,presleep_stage2_000,design_c_stage2_rolling_history,380,0.602486,0.662779,0.585455,0.530719,0.454139,0.638365,-0.031276



=== Test Row ===


,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
2,presleep_stage2_000,pre_sleep,design_c_stage2_rolling_history,mlp_current_day,1,test,2026,2,0.48,0.48,0.602486,0.530719,0.454139,0.638365,319,244,115,203,0.662779,0.585455



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [5]:
# Cell 1. Build compact Stage 2B rolling/history tensors
# 원하는 결과:
# - full Stage 2 rolling/history dataset에서 compact rolling feature만 선택한다.
# - leakage-safe feature만 포함되는지 점검한다.
# - train 기준 imputation/scaling/zero-variance 제거를 수행한다.
# - MLP용 train/validation/test tensor를 저장한다.
# - 아직 학습은 하지 않는다.

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
STAGE2_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage2_rolling_history"
STAGE2B_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage2b_compact_rolling"
STAGE2B_DIR.mkdir(parents=True, exist_ok=True)

STAGE2_DATASET_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_rolling_history_dataset.csv"
STAGE1_FINAL_FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"
STAGE2_FEATURE_COLUMNS_PATH = STAGE2_DIR / "pre_sleep_design_c_stage2_feature_columns.csv"

STAGE2B_DATASET_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_compact_rolling_dataset.csv"
STAGE2B_FEATURE_COLUMNS_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_feature_columns.csv"
STAGE2B_FINAL_FEATURE_COLUMNS_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_final_feature_columns.csv"
STAGE2B_ZERO_VARIANCE_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_zero_variance_removed_features.csv"
STAGE2B_MISSING_SUMMARY_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_missing_summary.csv"
STAGE2B_IMPUTER_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_median_imputer.joblib"
STAGE2B_SCALER_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_standard_scaler.joblib"
STAGE2B_TENSOR_DIR = STAGE2B_DIR / "mlp_current_day_final"
STAGE2B_TENSOR_SUMMARY_PATH = STAGE2B_DIR / "pre_sleep_design_c_stage2b_tensor_summary.csv"
STAGE2B_METADATA_PATH = STAGE2B_DIR / "metadata.json"

LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

STAGE2B_TENSOR_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "participant_object_id"
TARGET_COL = "good_sleep_label"
SPLIT_COL = "pre_sleep_split"
TIME_COL = "sleep_start_datetime"

stage2_full_df = pd.read_csv(STAGE2_DATASET_PATH, encoding="utf-8-sig")
stage1_final_feature_columns_df = pd.read_csv(STAGE1_FINAL_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")
stage2_feature_columns_df = pd.read_csv(STAGE2_FEATURE_COLUMNS_PATH, encoding="utf-8-sig")

stage2_full_df[TIME_COL] = pd.to_datetime(stage2_full_df[TIME_COL])
stage2_full_df = stage2_full_df.sort_values([ID_COL, TIME_COL]).reset_index(drop=True)

stage1_final_features = stage1_final_feature_columns_df["feature"].tolist()

compact_rolling_features = [
    feature
    for feature in stage2_full_df.columns
    if (
        feature.endswith("_prev_episode_diff1")
        or feature.endswith("_prev3_episode_mean")
        or feature.endswith("_dev_from_prev3_episode_mean")
    )
]

history_indicator_features = [
    "participant_episode_order",
    "has_prev3_episodes",
    "has_prev7_episodes",
]

stage2b_candidate_features = (
    stage1_final_features
    + compact_rolling_features
    + history_indicator_features
)

missing_candidate_features = [
    feature for feature in stage2b_candidate_features
    if feature not in stage2_full_df.columns
]

duplicate_candidate_count = (
    len(stage2b_candidate_features) - len(set(stage2b_candidate_features))
)

forbidden_name_patterns = [
    TARGET_COL,
    "minutesAsleep",
    "timeInBed",
    "sleep_duration",
    "efficiency",
    "sleep_score",
    "sleep_stage",
]

potential_leakage_features = [
    feature
    for feature in stage2b_candidate_features
    if any(pattern in feature for pattern in forbidden_name_patterns)
]

non_compact_rolling_features = [
    feature
    for feature in compact_rolling_features
    if (
        "prev7" in feature
        or feature.endswith("_std")
        or feature.endswith("_min")
        or feature.endswith("_max")
    )
]

stage2b_df = stage2_full_df.copy()

stage2b_feature_columns_df = pd.DataFrame(
    {
        "feature": stage2b_candidate_features,
        "feature_group": [
            (
                "stage1_final"
                if feature in stage1_final_features
                else "compact_rolling_history"
                if feature in compact_rolling_features
                else "history_indicator"
            )
            for feature in stage2b_candidate_features
        ],
    }
)

stage2b_missing_summary = (
    stage2b_df[stage2b_candidate_features]
    .isna()
    .mean()
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "feature"})
    .sort_values(["missing_rate", "feature"], ascending=[False, True])
)

train_mask = stage2b_df[SPLIT_COL] == "train"

X_raw = stage2b_df[stage2b_candidate_features].copy()
y = stage2b_df[TARGET_COL].astype(np.int64).to_numpy()

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_imputed = imputer.fit_transform(X_raw.loc[train_mask])
scaler.fit(X_train_imputed)

X_imputed = imputer.transform(X_raw)
X_scaled = scaler.transform(X_imputed).astype(np.float32)

train_scaled = X_scaled[train_mask.to_numpy()]
train_feature_std = train_scaled.std(axis=0)

zero_variance_mask = train_feature_std <= 1e-8

stage2b_zero_variance_features = [
    feature
    for feature, is_zero in zip(stage2b_candidate_features, zero_variance_mask)
    if is_zero
]

stage2b_final_features = [
    feature
    for feature, is_zero in zip(stage2b_candidate_features, zero_variance_mask)
    if not is_zero
]

X_scaled_final = X_scaled[:, ~zero_variance_mask].astype(np.float32)

stage2b_zero_variance_df = pd.DataFrame(
    {
        "removed_feature": stage2b_zero_variance_features,
        "train_scaled_std": train_feature_std[zero_variance_mask],
    }
)

stage2b_final_feature_columns_df = stage2b_feature_columns_df[
    stage2b_feature_columns_df["feature"].isin(stage2b_final_features)
].copy()

tensor_summary_rows = []

for split_name in ["train", "validation", "test"]:
    split_mask = stage2b_df[SPLIT_COL].to_numpy() == split_name

    X_split = X_scaled_final[split_mask]
    y_split = y[split_mask]
    episode_ids_split = stage2b_df.loc[split_mask, "sleep_episode_id"].astype(str).to_numpy()
    participant_ids_split = stage2b_df.loc[split_mask, ID_COL].astype(str).to_numpy()

    output_path = STAGE2B_TENSOR_DIR / f"{split_name}.npz"

    np.savez_compressed(
        output_path,
        X=X_split,
        y=y_split,
        sleep_episode_id=episode_ids_split,
        participant_object_id=participant_ids_split,
        feature_columns=np.array(stage2b_final_features, dtype=object),
    )

    tensor_summary_rows.append(
        {
            "tensor_type": "mlp_current_day_final",
            "split": split_name,
            "samples": int(X_split.shape[0]),
            "features": int(X_split.shape[1]),
            "participants": int(pd.Series(participant_ids_split).nunique()),
            "target_0": int((y_split == 0).sum()),
            "target_1": int((y_split == 1).sum()),
            "target_mean": float(y_split.mean()),
            "nan_count": int(np.isnan(X_split).sum()),
            "inf_count": int(np.isinf(X_split).sum()),
            "path": str(output_path.relative_to(PROJECT_ROOT)),
        }
    )

stage2b_tensor_summary_df = pd.DataFrame(tensor_summary_rows)

stage2b_scaled_check = pd.DataFrame(
    [
        {"metric": "rows", "value": X_scaled_final.shape[0]},
        {"metric": "stage1_final_features", "value": len(stage1_final_features)},
        {"metric": "compact_rolling_features", "value": len(compact_rolling_features)},
        {"metric": "history_indicator_features", "value": len(history_indicator_features)},
        {"metric": "candidate_features", "value": len(stage2b_candidate_features)},
        {"metric": "zero_variance_features", "value": len(stage2b_zero_variance_features)},
        {"metric": "final_features", "value": len(stage2b_final_features)},
        {"metric": "nan_count", "value": int(np.isnan(X_scaled_final).sum())},
        {"metric": "inf_count", "value": int(np.isinf(X_scaled_final).sum())},
        {
            "metric": "max_abs_train_scaled_mean",
            "value": float(np.abs(X_scaled_final[train_mask.to_numpy()].mean(axis=0)).max()),
        },
        {
            "metric": "min_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).min()),
        },
        {
            "metric": "max_train_scaled_std",
            "value": float(X_scaled_final[train_mask.to_numpy()].std(axis=0).max()),
        },
    ]
)

stage2b_df.to_csv(STAGE2B_DATASET_PATH, index=False, encoding="utf-8-sig")
stage2b_feature_columns_df.to_csv(STAGE2B_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
stage2b_final_feature_columns_df.to_csv(STAGE2B_FINAL_FEATURE_COLUMNS_PATH, index=False, encoding="utf-8-sig")
stage2b_zero_variance_df.to_csv(STAGE2B_ZERO_VARIANCE_PATH, index=False, encoding="utf-8-sig")
stage2b_missing_summary.to_csv(STAGE2B_MISSING_SUMMARY_PATH, index=False, encoding="utf-8-sig")
stage2b_tensor_summary_df.to_csv(STAGE2B_TENSOR_SUMMARY_PATH, index=False, encoding="utf-8-sig")

joblib.dump(imputer, STAGE2B_IMPUTER_PATH)
joblib.dump(scaler, STAGE2B_SCALER_PATH)

stage2b_metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_name": "pre_sleep_design_c_stage2b_compact_rolling",
    "source_stage2_dataset": str(STAGE2_DATASET_PATH.relative_to(PROJECT_ROOT)),
    "prediction_target": "good_sleep_label for sleep episode",
    "prediction_cutoff": "sleep_start_datetime",
    "strict_rule": "Compact rolling features come from prior-episode rolling features created with shift(1).",
    "compact_policy": [
        "stage1_final_features",
        "prev_episode_diff1",
        "prev3_episode_mean",
        "dev_from_prev3_episode_mean",
        "history_indicators",
    ],
    "excluded_rolling_policy": [
        "prev7 rolling features",
        "rolling std/min/max features",
    ],
    "rows": int(len(stage2b_df)),
    "participants": int(stage2b_df[ID_COL].nunique()),
    "candidate_feature_count": int(len(stage2b_candidate_features)),
    "zero_variance_removed_features": int(len(stage2b_zero_variance_features)),
    "final_feature_count": int(len(stage2b_final_features)),
    "leakage_checks": {
        "missing_candidate_features": len(missing_candidate_features),
        "duplicate_candidate_count": duplicate_candidate_count,
        "potential_leakage_features": potential_leakage_features,
        "non_compact_rolling_features": non_compact_rolling_features,
    },
    "outputs": {
        "dataset": str(STAGE2B_DATASET_PATH.relative_to(PROJECT_ROOT)),
        "feature_columns": str(STAGE2B_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "final_feature_columns": str(STAGE2B_FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT)),
        "zero_variance_removed_features": str(STAGE2B_ZERO_VARIANCE_PATH.relative_to(PROJECT_ROOT)),
        "missing_summary": str(STAGE2B_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "tensor_dir": str(STAGE2B_TENSOR_DIR.relative_to(PROJECT_ROOT)),
        "tensor_summary": str(STAGE2B_TENSOR_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "imputer": str(STAGE2B_IMPUTER_PATH.relative_to(PROJECT_ROOT)),
        "scaler": str(STAGE2B_SCALER_PATH.relative_to(PROJECT_ROOT)),
    },
}

STAGE2B_METADATA_PATH.write_text(
    json.dumps(stage2b_metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 2B compact rolling tensors

Created compact rolling/history tensors to reduce overfitting risk.

- Notebook: `notebooks/15_pre_sleep_stage2b_compact_rolling_training.ipynb`
- Dataset: `{STAGE2B_DATASET_PATH.relative_to(PROJECT_ROOT)}`
- Tensor dir: `{STAGE2B_TENSOR_DIR.relative_to(PROJECT_ROOT)}`
- Rows: {len(stage2b_df)}
- Participants: {stage2b_df[ID_COL].nunique()}
- Stage 1 final features: {len(stage1_final_features)}
- Compact rolling features: {len(compact_rolling_features)}
- History indicators: {len(history_indicator_features)}
- Candidate features: {len(stage2b_candidate_features)}
- Removed zero-variance features: {len(stage2b_zero_variance_features)}
- Final features: {len(stage2b_final_features)}
- NaN/Inf after final tensor creation: {int(np.isnan(X_scaled_final).sum())} / {int(np.isinf(X_scaled_final).sum())}
- Leakage guard: only prior-episode compact rolling features are used; no target-derived feature names detected: {len(potential_leakage_features) == 0}.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Stage 2B Compact Rolling Tensors Saved ===")
print("dataset:", STAGE2B_DATASET_PATH.relative_to(PROJECT_ROOT), STAGE2B_DATASET_PATH.exists())
print("feature columns:", STAGE2B_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE2B_FEATURE_COLUMNS_PATH.exists())
print("final feature columns:", STAGE2B_FINAL_FEATURE_COLUMNS_PATH.relative_to(PROJECT_ROOT), STAGE2B_FINAL_FEATURE_COLUMNS_PATH.exists())
print("zero variance:", STAGE2B_ZERO_VARIANCE_PATH.relative_to(PROJECT_ROOT), STAGE2B_ZERO_VARIANCE_PATH.exists())
print("missing summary:", STAGE2B_MISSING_SUMMARY_PATH.relative_to(PROJECT_ROOT), STAGE2B_MISSING_SUMMARY_PATH.exists())
print("metadata:", STAGE2B_METADATA_PATH.relative_to(PROJECT_ROOT), STAGE2B_METADATA_PATH.exists())

print("\n=== Leakage / Feature Policy Checks ===")
print("missing candidate features:", len(missing_candidate_features))
print("duplicate candidate features:", duplicate_candidate_count)
print("potential leakage features:", potential_leakage_features)
print("non-compact rolling features:", non_compact_rolling_features[:20])
print("non-compact rolling feature count:", len(non_compact_rolling_features))

print("\n=== Stage 2B Scaled Feature Check ===")
display(stage2b_scaled_check)

print("\n=== Stage 2B Feature Group Counts ===")
display(stage2b_final_feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Stage 2B Tensor Summary ===")
display(stage2b_tensor_summary_df)

print("\n=== Removed Zero-Variance Features ===")
display(stage2b_zero_variance_df)

print("\n=== Saved Tensor Files ===")
for split_name in ["train", "validation", "test"]:
    path = STAGE2B_TENSOR_DIR / f"{split_name}.npz"
    print(path.relative_to(PROJECT_ROOT), path.exists())

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Stage 2B Compact Rolling Tensors Saved ===
dataset: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\pre_sleep_design_c_stage2b_compact_rolling_dataset.csv True
feature columns: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\pre_sleep_design_c_stage2b_feature_columns.csv True
final feature columns: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\pre_sleep_design_c_stage2b_final_feature_columns.csv True
zero variance: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\pre_sleep_design_c_stage2b_zero_variance_removed_features.csv True
missing summary: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\pre_sleep_design_c_stage2b_missing_summary.csv True
metadata: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\metadata.json True

=== Leakage / Feature Policy Checks ===
missing candidate features: 0
duplicate candidate features: 0
potential leakage features: []


,metric,value
0,rows,3.551000e+03
1,stage1_final_features,5.800000e+01
2,compact_rolling_features,8.700000e+01
3,history_indicator_features,3.000000e+00
4,candidate_features,1.480000e+02
5,zero_variance_features,0.000000e+00
6,final_features,1.480000e+02
7,nan_count,0.000000e+00
8,inf_count,0.000000e+00
9,max_abs_train_scaled_mean,4.684212e-07



=== Stage 2B Feature Group Counts ===


,feature_group,features
0,compact_rolling_history,87
1,stage1_final,58
2,history_indicator,3



=== Stage 2B Tensor Summary ===


,tensor_type,split,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count,path
0,mlp_current_day_final,train,2323,148,46,1365,958,0.412398,0,0,data\processed\pre_sleep_forecasting\design_c_...
1,mlp_current_day_final,validation,347,148,9,225,122,0.351585,0,0,data\processed\pre_sleep_forecasting\design_c_...
2,mlp_current_day_final,test,881,148,14,563,318,0.360953,0,0,data\processed\pre_sleep_forecasting\design_c_...



=== Removed Zero-Variance Features ===


,removed_feature,train_scaled_std



=== Saved Tensor Files ===
data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\mlp_current_day_final\train.npz True
data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\mlp_current_day_final\validation.npz True
data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\mlp_current_day_final\test.npz True

log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [6]:
# Cell 2. Stage 2B compact rolling MLP training setup and utilities
# 원하는 결과:
# - Stage 2B compact rolling tensor를 로드한다.
# - Stage 1/Stage 2 reference 결과를 로드한다.
# - compact feature용 MLP와 학습 유틸리티를 정의한다.
# - 아직 학습은 하지 않는다.

import random
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

STAGE1_OUTPUT_DIR = (
    PROCESSED_DIR
    / "pre_sleep_forecasting"
    / "design_c_stage1"
    / "experiments"
    / "stage1_mlp_outputs"
)
STAGE1_METRICS_PATH = STAGE1_OUTPUT_DIR / "pre_sleep_stage1_mlp_metrics.csv"

STAGE2_OUTPUT_DIR = (
    PROCESSED_DIR
    / "pre_sleep_forecasting"
    / "design_c_stage2_rolling_history"
    / "experiments"
    / "stage2_mlp_outputs"
)
STAGE2_METRICS_PATH = STAGE2_OUTPUT_DIR / "pre_sleep_stage2_mlp_metrics.csv"

OUTPUT_DIR = STAGE2B_DIR / "experiments" / "stage2b_mlp_outputs"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_PRE_SLEEP_STAGE2B_TRAINING = False

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "presleep_stage2b_000"
MODEL_FAMILY = "mlp_current_day"
FEATURE_TIMING = "pre_sleep"
SUBSET_NAME = "design_c_stage2b_compact_rolling"
WINDOW = 1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def load_npz_split(split_name):
    path = STAGE2B_TENSOR_DIR / f"{split_name}.npz"
    data = np.load(path, allow_pickle=True)
    return {
        "path": path,
        "X": data["X"].astype(np.float32),
        "y": data["y"].astype(np.int64),
        "sleep_episode_id": data["sleep_episode_id"].astype(str),
        "participant_object_id": data["participant_object_id"].astype(str),
        "feature_columns": data["feature_columns"].astype(str),
    }

split_data = {
    split_name: load_npz_split(split_name)
    for split_name in ["train", "validation", "test"]
}

loaded_feature_columns = split_data["train"]["feature_columns"].tolist()

stage1_metrics_df = pd.read_csv(STAGE1_METRICS_PATH, encoding="utf-8-sig")
stage2_metrics_df = pd.read_csv(STAGE2_METRICS_PATH, encoding="utf-8-sig")

tensor_summary_rows = []

for split_name, data in split_data.items():
    X = data["X"]
    y = data["y"]
    participant_ids = data["participant_object_id"]

    tensor_summary_rows.append(
        {
            "split": split_name,
            "path": str(data["path"].relative_to(PROJECT_ROOT)),
            "X_shape": X.shape,
            "samples": int(X.shape[0]),
            "features": int(X.shape[1]),
            "participants": int(pd.Series(participant_ids).nunique()),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

tensor_summary_df = pd.DataFrame(tensor_summary_rows)

class PreSleepStage2BMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(96, 48, 24), dropout=0.30):
        super().__init__()

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(current_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

def make_loader(X, y, batch_size=64, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def predict_proba(model, X, batch_size=256):
    model.eval()
    probabilities = []

    loader = make_loader(X, np.zeros(X.shape[0]), batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch_X, _ in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)
            batch_probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            probabilities.append(batch_probabilities)

    return np.concatenate(probabilities)

def evaluate_binary_predictions(y_true, y_probability, threshold):
    y_pred = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probability)
        metrics["average_precision"] = average_precision_score(y_true, y_probability)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics

def find_best_threshold(y_true, y_probability, metric_name="balanced_accuracy"):
    threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []
    for threshold in threshold_grid:
        row = evaluate_binary_predictions(y_true, y_probability, threshold)
        rows.append(row)

    threshold_df = pd.DataFrame(rows)

    best_row = (
        threshold_df
        .sort_values(
            [metric_name, "f1", "balanced_accuracy"],
            ascending=[False, False, False],
        )
        .iloc[0]
        .to_dict()
    )

    return float(best_row["threshold"]), threshold_df, best_row

def train_one_stage2b_model(
    experiment_id,
    X_train,
    y_train,
    X_validation,
    y_validation,
    input_dim,
    seed=2026,
    max_epochs=120,
    patience=20,
    batch_size=64,
    learning_rate=8e-4,
    weight_decay=4e-4,
    hidden_dims=(96, 48, 24),
    dropout=0.30,
):
    set_seed(seed)

    model = PreSleepStage2BMLP(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(DEVICE)

    positive_count = float((y_train == 1).sum())
    negative_count = float((y_train == 0).sum())
    pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = None
    best_threshold = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        validation_probability = predict_proba(model, X_validation)
        threshold, threshold_df, best_threshold_row = find_best_threshold(
            y_validation,
            validation_probability,
            metric_name="balanced_accuracy",
        )

        validation_metrics = evaluate_binary_predictions(
            y_validation,
            validation_probability,
            threshold,
        )

        mean_train_loss = float(np.mean(train_losses))

        history_row = {
            "experiment_id": experiment_id,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "validation_selected_threshold": threshold,
            **{
                f"validation_{key}": value
                for key, value in validation_metrics.items()
            },
        }
        history_rows.append(history_row)

        current_validation_balanced_accuracy = validation_metrics["balanced_accuracy"]

        if current_validation_balanced_accuracy > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_validation_balanced_accuracy
            best_epoch = epoch
            best_threshold = threshold
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{experiment_id} epoch {epoch:03d} | "
                f"loss={mean_train_loss:.4f} | "
                f"val_bal_acc={validation_metrics['balanced_accuracy']:.4f} | "
                f"val_auc={validation_metrics['roc_auc']:.4f} | "
                f"thr={threshold:.2f}"
            )

        if epochs_without_improvement >= patience:
            print(
                f"{experiment_id} early stopped at epoch {epoch}; "
                f"best_epoch={best_epoch}, "
                f"best_val_bal_acc={best_validation_balanced_accuracy:.4f}"
            )
            break

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)

    return {
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
        "best_validation_balanced_accuracy": best_validation_balanced_accuracy,
    }

problem_rows = tensor_summary_df[
    (tensor_summary_df["nan_count"] > 0)
    | (tensor_summary_df["inf_count"] > 0)
]

print("=== Stage 2B Training Setup ===")
print("DEVICE:", DEVICE)
print("RUN_PRE_SLEEP_STAGE2B_TRAINING:", RUN_PRE_SLEEP_STAGE2B_TRAINING)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("TENSOR_DIR:", STAGE2B_TENSOR_DIR)

print("\n=== Tensor Summary ===")
display(tensor_summary_df)

print("\n=== Stage 1 / Stage 2 Test References ===")
display(
    pd.concat(
        [
            stage1_metrics_df.loc[stage1_metrics_df["split"] == "test"],
            stage2_metrics_df.loc[stage2_metrics_df["split"] == "test"],
        ],
        ignore_index=True,
    )[
        [
            "experiment_id",
            "subset_name",
            "split",
            "balanced_accuracy",
            "roc_auc",
            "average_precision",
            "f1",
            "precision",
            "recall",
        ]
    ]
)

print("\n=== Problem Rows ===")
if len(problem_rows) > 0:
    display(problem_rows)
else:
    print("없음")

print("\nStage 2B training utilities are ready.")
print("Model:", PreSleepStage2BMLP(input_dim=split_data["train"]["X"].shape[1]))

=== Stage 2B Training Setup ===
DEVICE: cpu
RUN_PRE_SLEEP_STAGE2B_TRAINING: False
EXPERIMENT_ID: presleep_stage2b_000
TENSOR_DIR: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\mlp_current_day_final

=== Tensor Summary ===


,split,path,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,train,data\processed\pre_sleep_forecasting\design_c_...,"(2323, 148)",2323,148,46,1365,958,0.412398,0,0
1,validation,data\processed\pre_sleep_forecasting\design_c_...,"(347, 148)",347,148,9,225,122,0.351585,0,0
2,test,data\processed\pre_sleep_forecasting\design_c_...,"(881, 148)",881,148,14,563,318,0.360953,0,0



=== Stage 1 / Stage 2 Test References ===


,experiment_id,subset_name,split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall
0,presleep_stage1_000,design_c_stage1_intraday_previous_day,test,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516
1,presleep_stage2_000,design_c_stage2_rolling_history,test,0.602486,0.662779,0.585455,0.530719,0.454139,0.638365



=== Problem Rows ===
없음

Stage 2B training utilities are ready.
Model: PreSleepStage2BMLP(
  (network): Sequential(
    (0): Linear(in_features=148, out_features=96, bias=True)
    (1): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=96, out_features=48, bias=True)
    (5): BatchNorm1d(48, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=48, out_features=24, bias=True)
    (9): BatchNorm1d(24, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=24, out_features=1, bias=True)
  )
)


In [8]:
# Cell 3. Train Pre-sleep Design C Stage 2B compact rolling MLP
# 원하는 결과:
# - Stage 2B compact rolling MLP를 학습한다.
# - validation balanced accuracy 기준 best epoch/threshold를 선택한다.
# - validation-selected threshold로 train/validation/test metrics를 저장한다.
# - Stage 1/Stage 2/Stage 2B test 결과를 비교한다.
# - prediction/history/model 파일을 저장하고 log를 업데이트한다.

RUN_PRE_SLEEP_STAGE2B_TRAINING = True

MAX_EPOCHS = 120
PATIENCE = 20
BATCH_SIZE = 64
LEARNING_RATE = 8e-4
WEIGHT_DECAY = 4e-4
HIDDEN_DIMS = (96, 48, 24)
DROPOUT = 0.30

METRICS_PATH = OUTPUT_DIR / "pre_sleep_stage2b_mlp_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "pre_sleep_stage2b_mlp_training_history.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "pre_sleep_stage2b_mlp_predictions.csv"
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_ID}_best.pt"
COMPARISON_PATH = OUTPUT_DIR / "pre_sleep_stage1_stage2_stage2b_comparison.csv"

if not RUN_PRE_SLEEP_STAGE2B_TRAINING:
    print("RUN_PRE_SLEEP_STAGE2B_TRAINING is False.")
    print("Set RUN_PRE_SLEEP_STAGE2B_TRAINING = True to train manually.")
else:
    print("=== Pre-Sleep Stage 2B Compact Rolling MLP Training Started ===")
    print("device:", DEVICE)
    print("experiment_id:", EXPERIMENT_ID)
    print("max_epochs:", MAX_EPOCHS)
    print("patience:", PATIENCE)
    print("batch_size:", BATCH_SIZE)
    print("learning_rate:", LEARNING_RATE)
    print("weight_decay:", WEIGHT_DECAY)
    print("hidden_dims:", HIDDEN_DIMS)
    print("dropout:", DROPOUT)

    train_result = train_one_stage2b_model(
        experiment_id=EXPERIMENT_ID,
        X_train=split_data["train"]["X"],
        y_train=split_data["train"]["y"],
        X_validation=split_data["validation"]["X"],
        y_validation=split_data["validation"]["y"],
        input_dim=split_data["train"]["X"].shape[1],
        seed=SEED,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    )

    model = train_result["model"]
    best_epoch = train_result["best_epoch"]
    selected_threshold = train_result["best_threshold"]
    history_df = train_result["history_df"]

    torch.save(
        {
            "experiment_id": EXPERIMENT_ID,
            "model_family": MODEL_FAMILY,
            "feature_timing": FEATURE_TIMING,
            "subset_name": SUBSET_NAME,
            "window": WINDOW,
            "seed": SEED,
            "input_dim": split_data["train"]["X"].shape[1],
            "hidden_dims": HIDDEN_DIMS,
            "dropout": DROPOUT,
            "best_epoch": best_epoch,
            "selected_threshold_from_validation": selected_threshold,
            "state_dict": model.state_dict(),
            "feature_columns": loaded_feature_columns,
        },
        MODEL_PATH,
    )

    metrics_rows = []
    prediction_rows = []

    for split_name in ["train", "validation", "test"]:
        X_split = split_data[split_name]["X"]
        y_split = split_data[split_name]["y"]
        probability = predict_proba(model, X_split)

        metrics = evaluate_binary_predictions(
            y_split,
            probability,
            selected_threshold,
        )

        metrics_rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "feature_timing": FEATURE_TIMING,
                "subset_name": SUBSET_NAME,
                "model_family": MODEL_FAMILY,
                "window": WINDOW,
                "split": split_name,
                "seed": SEED,
                "best_epoch": best_epoch,
                "selected_threshold_from_validation": selected_threshold,
                **metrics,
            }
        )

        split_prediction_df = pd.DataFrame(
            {
                "experiment_id": EXPERIMENT_ID,
                "split": split_name,
                "sleep_episode_id": split_data[split_name]["sleep_episode_id"],
                "participant_object_id": split_data[split_name]["participant_object_id"],
                "y_true": y_split,
                "y_probability": probability,
                "selected_threshold_from_validation": selected_threshold,
                "y_pred": (probability >= selected_threshold).astype(int),
            }
        )
        prediction_rows.append(split_prediction_df)

    metrics_df = pd.DataFrame(metrics_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    stage1_test_reference = stage1_metrics_df[
        stage1_metrics_df["split"] == "test"
    ].iloc[0]

    stage2_test_reference = stage2_metrics_df[
        stage2_metrics_df["split"] == "test"
    ].iloc[0]

    stage2b_test_row = metrics_df[metrics_df["split"] == "test"].iloc[0]

    comparison_df = pd.DataFrame(
        [
            {
                "model_role": "stage1_reference",
                "experiment_id": stage1_test_reference["experiment_id"],
                "subset_name": stage1_test_reference["subset_name"],
                "features": 58,
                "test_balanced_accuracy": stage1_test_reference["balanced_accuracy"],
                "test_roc_auc": stage1_test_reference["roc_auc"],
                "test_average_precision": stage1_test_reference["average_precision"],
                "test_f1": stage1_test_reference["f1"],
                "test_precision": stage1_test_reference["precision"],
                "test_recall": stage1_test_reference["recall"],
            },
            {
                "model_role": "stage2_full_rolling",
                "experiment_id": stage2_test_reference["experiment_id"],
                "subset_name": stage2_test_reference["subset_name"],
                "features": 380,
                "test_balanced_accuracy": stage2_test_reference["balanced_accuracy"],
                "test_roc_auc": stage2_test_reference["roc_auc"],
                "test_average_precision": stage2_test_reference["average_precision"],
                "test_f1": stage2_test_reference["f1"],
                "test_precision": stage2_test_reference["precision"],
                "test_recall": stage2_test_reference["recall"],
            },
            {
                "model_role": "stage2b_compact_rolling",
                "experiment_id": EXPERIMENT_ID,
                "subset_name": SUBSET_NAME,
                "features": split_data["train"]["X"].shape[1],
                "test_balanced_accuracy": stage2b_test_row["balanced_accuracy"],
                "test_roc_auc": stage2b_test_row["roc_auc"],
                "test_average_precision": stage2b_test_row["average_precision"],
                "test_f1": stage2b_test_row["f1"],
                "test_precision": stage2b_test_row["precision"],
                "test_recall": stage2b_test_row["recall"],
            },
        ]
    )

    comparison_df["delta_balanced_accuracy_vs_stage1"] = (
        comparison_df["test_balanced_accuracy"]
        - float(stage1_test_reference["balanced_accuracy"])
    )
    comparison_df["delta_balanced_accuracy_vs_stage2_full"] = (
        comparison_df["test_balanced_accuracy"]
        - float(stage2_test_reference["balanced_accuracy"])
    )

    metrics_df.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
    history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
    comparison_df.to_csv(COMPARISON_PATH, index=False, encoding="utf-8-sig")

    stage2b_metadata["stage2b_training"] = {
        "experiment_id": EXPERIMENT_ID,
        "model_family": MODEL_FAMILY,
        "feature_timing": FEATURE_TIMING,
        "subset_name": SUBSET_NAME,
        "window": WINDOW,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "hidden_dims": list(HIDDEN_DIMS),
        "dropout": DROPOUT,
        "best_epoch": int(best_epoch),
        "selected_threshold_from_validation": float(selected_threshold),
        "metrics_path": str(METRICS_PATH.relative_to(PROJECT_ROOT)),
        "history_path": str(HISTORY_PATH.relative_to(PROJECT_ROOT)),
        "predictions_path": str(PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
        "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
        "comparison_path": str(COMPARISON_PATH.relative_to(PROJECT_ROOT)),
    }
    STAGE2B_METADATA_PATH.write_text(
        json.dumps(stage2b_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 2B compact rolling MLP training

Trained compact rolling/history strict pre-sleep forecasting MLP.

- Notebook: `notebooks/15_pre_sleep_stage2b_compact_rolling_training.ipynb`
- Experiment: `{EXPERIMENT_ID}`
- Objective: predict `good_sleep_label` using pre-sleep features plus compact prior-episode rolling/history features
- Model: `{MODEL_FAMILY}`
- Features: {split_data["train"]["X"].shape[1]}
- Train/validation/test samples: {split_data["train"]["X"].shape[0]} / {split_data["validation"]["X"].shape[0]} / {split_data["test"]["X"].shape[0]}
- Best epoch: {best_epoch}
- Selected threshold from validation: {selected_threshold:.2f}
- Test balanced accuracy: {stage2b_test_row["balanced_accuracy"]:.4f}
- Test ROC AUC: {stage2b_test_row["roc_auc"]:.4f}
- Test average precision: {stage2b_test_row["average_precision"]:.4f}
- Test F1: {stage2b_test_row["f1"]:.4f}
- Test precision: {stage2b_test_row["precision"]:.4f}
- Test recall: {stage2b_test_row["recall"]:.4f}
- Delta test balanced accuracy vs Stage 1 single-seed: {stage2b_test_row["balanced_accuracy"] - float(stage1_test_reference["balanced_accuracy"]):.4f}
- Delta test balanced accuracy vs Stage 2 full rolling: {stage2b_test_row["balanced_accuracy"] - float(stage2_test_reference["balanced_accuracy"]):.4f}
- Metrics: `{METRICS_PATH.relative_to(PROJECT_ROOT)}`
- History: `{HISTORY_PATH.relative_to(PROJECT_ROOT)}`
- Predictions: `{PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}`
- Comparison: `{COMPARISON_PATH.relative_to(PROJECT_ROOT)}`
- Model: `{MODEL_PATH.relative_to(PROJECT_ROOT)}`
"""

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(log_entry)

    print("\n=== Pre-Sleep Stage 2B Compact Rolling MLP Training Finished ===")
    print("model:", MODEL_PATH.relative_to(PROJECT_ROOT), MODEL_PATH.exists())
    print("metrics:", METRICS_PATH.relative_to(PROJECT_ROOT), METRICS_PATH.exists())
    print("history:", HISTORY_PATH.relative_to(PROJECT_ROOT), HISTORY_PATH.exists())
    print("predictions:", PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PREDICTIONS_PATH.exists())
    print("comparison:", COMPARISON_PATH.relative_to(PROJECT_ROOT), COMPARISON_PATH.exists())

    print("\n=== Metrics ===")
    display(metrics_df)

    print("\n=== Stage 1 vs Stage 2 vs Stage 2B Test Comparison ===")
    display(comparison_df)

    print("\n=== Test Row ===")
    display(metrics_df.loc[metrics_df["split"] == "test"])

    print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Pre-Sleep Stage 2B Compact Rolling MLP Training Started ===
device: cpu
experiment_id: presleep_stage2b_000
max_epochs: 120
patience: 20
batch_size: 64
learning_rate: 0.0008
weight_decay: 0.0004
hidden_dims: (96, 48, 24)
dropout: 0.3
presleep_stage2b_000 epoch 001 | loss=0.8170 | val_bal_acc=0.6566 | val_auc=0.6920 | thr=0.52
presleep_stage2b_000 epoch 010 | loss=0.6960 | val_bal_acc=0.6917 | val_auc=0.7286 | thr=0.42
presleep_stage2b_000 epoch 020 | loss=0.6116 | val_bal_acc=0.6772 | val_auc=0.7134 | thr=0.37
presleep_stage2b_000 early stopped at epoch 29; best_epoch=9, best_val_bal_acc=0.7123

=== Pre-Sleep Stage 2B Compact Rolling MLP Training Finished ===
model: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\experiments\stage2b_mlp_outputs\models\presleep_stage2b_000_best.pt True
metrics: data\processed\pre_sleep_forecasting\design_c_stage2b_compact_rolling\experiments\stage2b_mlp_outputs\pre_sleep_stage2b_mlp_metrics.csv True
history: data\processed\pre_

,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
0,presleep_stage2b_000,pre_sleep,design_c_stage2b_compact_rolling,mlp_current_day,1,train,2026,9,0.4,0.4,0.692295,0.681873,0.552888,0.889353,676,689,106,852,0.805487,0.749559
1,presleep_stage2b_000,pre_sleep,design_c_stage2b_compact_rolling,mlp_current_day,1,validation,2026,9,0.4,0.4,0.712350,0.649275,0.502242,0.918033,114,111,10,112,0.735009,0.551719
2,presleep_stage2b_000,pre_sleep,design_c_stage2b_compact_rolling,mlp_current_day,1,test,2026,9,0.4,0.4,0.592293,0.553005,0.423786,0.795597,219,344,65,253,0.685227,0.578818



=== Stage 1 vs Stage 2 vs Stage 2B Test Comparison ===


,model_role,experiment_id,subset_name,features,test_balanced_accuracy,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall,delta_balanced_accuracy_vs_stage1,delta_balanced_accuracy_vs_stage2_full
0,stage1_reference,presleep_stage1_000,design_c_stage1_intraday_previous_day,58,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516,0.000000,0.031276
1,stage2_full_rolling,presleep_stage2_000,design_c_stage2_rolling_history,380,0.602486,0.662779,0.585455,0.530719,0.454139,0.638365,-0.031276,0.000000
2,stage2b_compact_rolling,presleep_stage2b_000,design_c_stage2b_compact_rolling,148,0.592293,0.685227,0.578818,0.553005,0.423786,0.795597,-0.041470,-0.010194



=== Test Row ===


,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
2,presleep_stage2b_000,pre_sleep,design_c_stage2b_compact_rolling,mlp_current_day,1,test,2026,9,0.4,0.4,0.592293,0.553005,0.423786,0.795597,219,344,65,253,0.685227,0.578818



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
